# Dog Locomotion — JAX/MJX + Brax PPO

Trains a joystick locomotion policy for the **custom 2 kg dog quadruped** using
MuJoCo Playground's training stack (JAX + MJX + Brax PPO) — the same approach that
trains the Unitree Go1 in ~7 minutes on an RTX 4090.

**Requires a GPU runtime.** Runtime → Change runtime type → GPU (T4 is fine).

The environment is a self-contained port of Playground's `Go1JoystickFlatTerrain`
task, adapted to this robot's body/joint/sensor names.

In [ ]:
# @title Install JAX, MuJoCo, MuJoCo Playground (then RESTART runtime)
!pip uninstall -y jax jaxlib brax mujoco mujoco-mjx mujoco_mjx mujoco-playground warp-lang playground

!pip install "jax[cuda12]==0.6.2"
!pip install git+https://github.com/google-deepmind/mujoco.git
!pip install git+https://github.com/google-deepmind/mujoco_playground.git

print("Install complete. Now RESTART the runtime (Runtime > Restart), then run the next cell.")

In [ ]:
# @title GPU check + EGL rendering setup
import os, subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError('No GPU. Runtime > Change runtime type > GPU.')

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ['MUJOCO_GL'] = 'egl'
xla_flags = os.environ.get('XLA_FLAGS', '')
os.environ['XLA_FLAGS'] = xla_flags + ' --xla_gpu_triton_gemm_any=True'

import mujoco
mujoco.MjModel.from_xml_string('<mujoco/>')
print('MuJoCo OK')

In [ ]:
# @title Clone the dog model repo (provides assets/dog_scene.xml + meshes)
import os
if not os.path.exists('quadruped_rl'):
  !git clone https://github.com/nbaron17/quadruped_rl.git
SCENE_PATH = 'quadruped_rl/assets/dog_scene.xml'
assert os.path.exists(SCENE_PATH), SCENE_PATH
print('Scene:', SCENE_PATH)

In [ ]:
# @title Imports
import functools
from datetime import datetime
from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
import numpy as np
import mujoco
from mujoco import mjx
from ml_collections import config_dict
import matplotlib.pyplot as plt
import mediapy as media
from IPython.display import clear_output, display

from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from mujoco_playground._src import mjx_env
from mujoco_playground import wrapper

print('JAX devices:', jax.devices())

## The environment

A faithful port of Playground's Go1 joystick task. The policy outputs joint-position
offsets from the default pose; a position actuator (Kp) tracks them. The robot is
rewarded for tracking commanded forward/lateral/yaw velocity, with the usual
regularisation costs (orientation, energy, action rate, foot air-time, etc.).

In [ ]:
# @title Helper + config

FEET = ["foot_FR", "foot_FL", "foot_BR", "foot_BL"]


def get_sensor(model, data, name):
  sid = model.sensor(name).id
  adr = model.sensor_adr[sid]
  dim = model.sensor_dim[sid]
  return data.sensordata[adr:adr + dim]


def default_config():
  return config_dict.create(
      ctrl_dt=0.02,
      sim_dt=0.004,
      episode_length=1000,
      Kp=20.0,
      Kd=0.5,
      action_scale=0.5,
      # MJX backend + buffer caps (mirrors Go1's tuning; speeds up compile + steps)
      impl="jax",
      naconmax=16,
      njmax=40,
      soft_joint_pos_limit_factor=0.95,
      noise_config=config_dict.create(
          level=1.0,
          scales=config_dict.create(
              joint_pos=0.03, joint_vel=1.5, gyro=0.2, gravity=0.05, linvel=0.1,
          ),
      ),
      reward_config=config_dict.create(
          scales=config_dict.create(
              tracking_lin_vel=1.0,
              tracking_ang_vel=0.5,
              lin_vel_z=-0.5,
              ang_vel_xy=-0.05,
              orientation=-5.0,
              dof_pos_limits=-1.0,
              pose=0.5,
              termination=-1.0,
              stand_still=-1.0,
              torques=-0.0002,
              action_rate=-0.01,
              energy=-0.001,
              feet_clearance=-2.0,
              feet_height=-0.2,
              feet_slip=-0.1,
              feet_air_time=0.1,
          ),
          tracking_sigma=0.25,
          max_foot_height=0.06,
      ),
      # Command amplitude [vx, vy, yaw] and probability of a non-zero command.
      command_config=config_dict.create(a=[0.6, 0.4, 1.0], b=[0.9, 0.25, 0.5]),
  )

In [ ]:
# @title DogJoystickEnv

class DogJoystickEnv(mjx_env.MjxEnv):
  def __init__(self, xml_path, config=None, config_overrides=None):
    super().__init__(config or default_config(), config_overrides)
    self._mj_model = mujoco.MjModel.from_xml_path(xml_path)
    self._mj_model.opt.timestep = self._config.sim_dt
    self._mj_model.opt.ccd_iterations = 20
    # Position-servo PD gains.
    self._mj_model.dof_damping[6:] = self._config.Kd
    self._mj_model.actuator_gainprm[:, 0] = self._config.Kp
    self._mj_model.actuator_biasprm[:, 1] = -self._config.Kp
    self._mjx_model = mjx.put_model(self._mj_model, impl=self._config.impl)
    self._xml_path = xml_path
    self._post_init()

  def _post_init(self):
    self._init_q = jp.array(self._mj_model.keyframe("home").qpos)
    self._default_pose = jp.array(self._mj_model.keyframe("home").qpos[7:])
    self._lowers, self._uppers = self._mj_model.jnt_range[1:].T
    f = self._config.soft_joint_pos_limit_factor
    self._soft_lowers, self._soft_uppers = self._lowers * f, self._uppers * f
    self._torso_body_id = self._mj_model.body("base").id
    self._torso_mass = self._mj_model.body_subtreemass[self._torso_body_id]
    self._imu_site_id = self._mj_model.site("imu").id
    self._feet_site_id = np.array([self._mj_model.site(s).id for s in FEET])
    adr = []
    for s in FEET:
      sid = self._mj_model.sensor(f"{s}_global_linvel").id
      a = self._mj_model.sensor_adr[sid]
      adr.append(list(range(a, a + self._mj_model.sensor_dim[sid])))
    self._foot_linvel_sensor_adr = jp.array(adr)
    self._feet_floor_found_sensor = [
        self._mj_model.sensor(f"{s}_floor_found").id for s in FEET]
    self._cmd_a = jp.array(self._config.command_config.a)
    self._cmd_b = jp.array(self._config.command_config.b)

  # --- properties required by MjxEnv ---
  @property
  def xml_path(self): return self._xml_path
  @property
  def action_size(self): return self._mjx_model.nu
  @property
  def mj_model(self): return self._mj_model
  @property
  def mjx_model(self): return self._mjx_model

  # --- sensor helpers ---
  def get_gravity(self, data):
    return data.site_xmat[self._imu_site_id].reshape(3, 3).T @ jp.array([0, 0, -1.0])
  def get_gyro(self, data): return get_sensor(self.mj_model, data, "gyro")
  def get_local_linvel(self, data): return get_sensor(self.mj_model, data, "local_linvel")
  def get_accelerometer(self, data): return get_sensor(self.mj_model, data, "accelerometer")
  def get_global_linvel(self, data): return get_sensor(self.mj_model, data, "global_linvel")
  def get_global_angvel(self, data): return get_sensor(self.mj_model, data, "global_angvel")
  def get_upvector(self, data): return get_sensor(self.mj_model, data, "upvector")

  def _mjx_step_n(self, data, ctrl):
    data = data.replace(ctrl=ctrl)
    def f(d, _): return mjx.step(self._mjx_model, d), None
    data, _ = jax.lax.scan(f, data, None, self.n_substeps)
    return data

  def reset(self, rng):
    rng, k1, k2, k3 = jax.random.split(rng, 4)
    qpos = self._init_q
    qpos = qpos.at[0:2].set(qpos[0:2] + jax.random.uniform(k1, (2,), minval=-0.3, maxval=0.3))
    yaw = jax.random.uniform(k2, (1,), minval=-3.14, maxval=3.14)
    from mujoco.mjx._src import math as mjxmath
    quat = mjxmath.axis_angle_to_quat(jp.array([0, 0, 1.0]), yaw)
    qpos = qpos.at[3:7].set(mjxmath.quat_mul(qpos[3:7], quat))
    qvel = jp.zeros(self.mjx_model.nv)
    qvel = qvel.at[0:6].set(jax.random.uniform(k3, (6,), minval=-0.3, maxval=0.3))

    data = mjx.make_data(self._mjx_model, impl=self._config.impl,
                         naconmax=self._config.naconmax, njmax=self._config.njmax)
    data = data.replace(qpos=qpos, qvel=qvel, ctrl=self._default_pose)
    data = mjx.forward(self._mjx_model, data)

    rng, kc, kt = jax.random.split(rng, 3)
    cmd = jax.random.uniform(kc, (3,), minval=-self._cmd_a, maxval=self._cmd_a)
    steps_until_next_cmd = jp.round(jax.random.exponential(kt) * 5.0 / self.dt).astype(jp.int32)
    info = {
        "rng": rng, "command": cmd,
        "steps_until_next_cmd": steps_until_next_cmd,
        "last_act": jp.zeros(self.mjx_model.nu),
        "last_last_act": jp.zeros(self.mjx_model.nu),
        "feet_air_time": jp.zeros(4),
        "last_contact": jp.zeros(4, dtype=bool),
        "swing_peak": jp.zeros(4),
    }
    metrics = {f"reward/{k}": jp.zeros(()) for k in self._config.reward_config.scales}
    metrics["swing_peak"] = jp.zeros(())
    obs = self._get_obs(data, info)
    return mjx_env.State(data, obs, jp.zeros(()), jp.zeros(()), metrics, info)

  def step(self, state, action):
    motor_targets = self._default_pose + action * self._config.action_scale
    data = self._mjx_step_n(state.data, motor_targets)

    contact = jp.array([
        data.sensordata[self._mj_model.sensor_adr[s]] > 0
        for s in self._feet_floor_found_sensor])
    contact_filt = contact | state.info["last_contact"]
    first_contact = (state.info["feet_air_time"] > 0.0) * contact_filt
    state.info["feet_air_time"] += self.dt
    p_fz = data.site_xpos[self._feet_site_id][..., -1]
    state.info["swing_peak"] = jp.maximum(state.info["swing_peak"], p_fz)

    obs = self._get_obs(data, state.info)
    done = self._get_termination(data)
    rewards = self._get_reward(data, action, state.info, done, first_contact, contact)
    rewards = {k: v * self._config.reward_config.scales[k] for k, v in rewards.items()}
    reward = jp.clip(sum(rewards.values()) * self.dt, 0.0, 10000.0)

    state.info["last_last_act"] = state.info["last_act"]
    state.info["last_act"] = action
    state.info["steps_until_next_cmd"] -= 1
    state.info["rng"], k1, k2 = jax.random.split(state.info["rng"], 3)
    state.info["command"] = jp.where(
        state.info["steps_until_next_cmd"] <= 0,
        self.sample_command(k1, state.info["command"]), state.info["command"])
    state.info["steps_until_next_cmd"] = jp.where(
        (done > 0) | (state.info["steps_until_next_cmd"] <= 0),
        jp.round(jax.random.exponential(k2) * 5.0 / self.dt).astype(jp.int32),
        state.info["steps_until_next_cmd"])
    state.info["feet_air_time"] *= ~contact
    state.info["last_contact"] = contact
    state.info["swing_peak"] *= ~contact
    for k, v in rewards.items():
      state.metrics[f"reward/{k}"] = v
    state.metrics["swing_peak"] = jp.mean(state.info["swing_peak"])

    return state.replace(data=data, obs=obs, reward=reward, done=done.astype(reward.dtype))

  def _get_termination(self, data):
    return (self.get_upvector(data)[-1] < 0.0).astype(jp.float32)

  def _get_obs(self, data, info):
    def noisy(x, scale, rng):
      return x + (2 * jax.random.uniform(rng, x.shape) - 1) * self._config.noise_config.level * scale
    nc = self._config.noise_config.scales
    info["rng"], *ks = jax.random.split(info["rng"], 6)
    gyro = noisy(self.get_gyro(data), nc.gyro, ks[0])
    gravity = noisy(self.get_gravity(data), nc.gravity, ks[1])
    jang = noisy(data.qpos[7:], nc.joint_pos, ks[2]) - self._default_pose
    jvel = noisy(data.qvel[6:], nc.joint_vel, ks[3])
    linvel = noisy(self.get_local_linvel(data), nc.linvel, ks[4])

    state = jp.hstack([linvel, gyro, gravity, jang, jvel, info["last_act"], info["command"]])
    feet_vel = data.sensordata[self._foot_linvel_sensor_adr].ravel()
    privileged = jp.hstack([
        state, self.get_gyro(data), self.get_accelerometer(data), self.get_gravity(data),
        self.get_local_linvel(data), self.get_global_angvel(data),
        data.qpos[7:] - self._default_pose, data.qvel[6:], data.actuator_force,
        info["last_contact"], feet_vel, info["feet_air_time"]])
    return {"state": state, "privileged_state": privileged}

  def _get_reward(self, data, action, info, done, first_contact, contact):
    cmd = info["command"]
    cmd_norm = jp.linalg.norm(cmd)
    feet_vel = data.sensordata[self._foot_linvel_sensor_adr]
    vel_xy = feet_vel[..., :2]
    foot_z = data.site_xpos[self._feet_site_id][..., -1]
    mfh = self._config.reward_config.max_foot_height
    sigma = self._config.reward_config.tracking_sigma

    lin_err = jp.sum(jp.square(cmd[:2] - self.get_local_linvel(data)[:2]))
    ang_err = jp.square(cmd[2] - self.get_gyro(data)[2])
    out_lo = -jp.clip(data.qpos[7:] - self._soft_lowers, None, 0.0)
    out_hi = jp.clip(data.qpos[7:] - self._soft_uppers, 0.0, None)
    swing_err = info["swing_peak"] / mfh - 1.0

    return {
        "tracking_lin_vel": jp.exp(-lin_err / sigma),
        "tracking_ang_vel": jp.exp(-ang_err / sigma),
        "lin_vel_z": jp.square(self.get_global_linvel(data)[2]),
        "ang_vel_xy": jp.sum(jp.square(self.get_global_angvel(data)[:2])),
        "orientation": jp.sum(jp.square(self.get_upvector(data)[:2])),
        "dof_pos_limits": jp.sum(out_lo + out_hi),
        "pose": jp.exp(-jp.sum(jp.square(data.qpos[7:] - self._default_pose) * jp.array([1.0, 1.0, 0.1] * 4))),
        "stand_still": jp.sum(jp.abs(data.qpos[7:] - self._default_pose)) * (cmd_norm < 0.01),
        "termination": done,
        "torques": jp.sqrt(jp.sum(jp.square(data.actuator_force))) + jp.sum(jp.abs(data.actuator_force)),
        "action_rate": jp.sum(jp.square(action - info["last_act"])),
        "energy": jp.sum(jp.abs(data.qvel[6:]) * jp.abs(data.actuator_force)),
        "feet_slip": jp.sum(jp.sum(jp.square(vel_xy), axis=-1) * contact) * (cmd_norm > 0.01),
        "feet_clearance": jp.sum(jp.abs(foot_z - mfh) * jp.sqrt(jp.linalg.norm(vel_xy, axis=-1))),
        "feet_height": jp.sum(jp.square(swing_err) * first_contact) * (cmd_norm > 0.01),
        "feet_air_time": jp.sum((info["feet_air_time"] - 0.1) * first_contact) * (cmd_norm > 0.01),
    }

  def sample_command(self, rng, x_k):
    rng, ry, rw, rz = jax.random.split(rng, 4)
    y_k = jax.random.uniform(ry, (3,), minval=-self._cmd_a, maxval=self._cmd_a)
    z_k = jax.random.bernoulli(rz, self._cmd_b, (3,))
    w_k = jax.random.bernoulli(rw, 0.5, (3,))
    return x_k - w_k * (x_k - y_k * z_k)

print("DogJoystickEnv defined")

In [ ]:
# @title Sanity-check the env steps
env = DogJoystickEnv(SCENE_PATH)
rng = jax.random.PRNGKey(0)
state = jax.jit(env.reset)(rng)
print("obs[state] dim:", state.obs["state"].shape)
print("obs[privileged] dim:", state.obs["privileged_state"].shape)
state = jax.jit(env.step)(state, jp.zeros(env.action_size))
print("reward after 1 step:", float(state.reward))
print("OK")

## Train

PPO config mirrors Playground's Go1 joystick. `num_timesteps` is set lower than the
Go1 default for a quick first run — raise it if the gait isn't clean. Lower `num_envs`
if you hit GPU out-of-memory.

In [ ]:
# @title PPO config + train
ppo_params = config_dict.create(
    num_timesteps=100_000_000,
    num_evals=10,
    reward_scaling=1.0,
    episode_length=1000,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=4,
    discounting=0.97,
    learning_rate=3e-4,
    entropy_cost=1e-2,
    num_envs=8192,
    batch_size=256,
    max_grad_norm=1.0,
    network_factory=config_dict.create(
        policy_hidden_layer_sizes=(512, 256, 128),
        value_hidden_layer_sizes=(512, 256, 128),
        policy_obs_key="state",
        value_obs_key="privileged_state",
    ),
)

x_data, y_data, y_err, times = [], [], [], [datetime.now()]
def progress(step, metrics):
  clear_output(wait=True)
  times.append(datetime.now())
  x_data.append(step)
  y_data.append(metrics["eval/episode_reward"])
  y_err.append(metrics["eval/episode_reward_std"])
  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("environment steps"); plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.2f}")
  plt.errorbar(x_data, y_data, yerr=y_err, color="blue")
  display(plt.gcf())

train_params = dict(ppo_params)
nf = train_params.pop("network_factory")
network_factory = functools.partial(ppo_networks.make_ppo_networks, **nf)

train_fn = functools.partial(
    ppo.train, **train_params,
    network_factory=network_factory,
    progress_fn=progress,
)

make_inference_fn, params, _ = train_fn(
    environment=DogJoystickEnv(SCENE_PATH),
    eval_env=DogJoystickEnv(SCENE_PATH),
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
print(f"jit: {times[1]-times[0]}  train: {times[-1]-times[1]}")

## Rollout & render

In [ ]:
# @title Render a rollout with a velocity command
x_vel, y_vel, yaw_vel = 0.4, 0.0, 0.0  # @param

eval_env = DogJoystickEnv(SCENE_PATH)
jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_inf = jax.jit(make_inference_fn(params, deterministic=True))

rng = jax.random.PRNGKey(1)
state = jit_reset(rng)
cmd = jp.array([x_vel, y_vel, yaw_vel])
rollout = []
for _ in range(500):
  state.info["command"] = cmd
  act, _ = jit_inf(state.obs, rng)
  rng, _ = jax.random.split(rng)
  state = jit_step(state, act)
  rollout.append(state)

fps = 1.0 / eval_env.dt
frames = eval_env.render(rollout, camera="track", height=480, width=640)
media.show_video(frames, fps=fps)